# **Объяснение агента: вклад инструментов и шагов**

Практика к модулю [«Объяснение поведения агентов»](https://ai-interpretability.school).

Настоящий прогон агента стоит денег и требует ключей, а ещё он недетерминирован — два запуска
дадут разные трейсы. Для практики это плохо: непонятно, что мы измеряем — метод или дисперсию
модели. Поэтому агент здесь **симулятор с известным устройством**: мы заранее знаем, какой
инструмент на что влияет, и проверяем по этому знанию сам метод.

Приём не выдуманный: так проверяют любое объяснение — сначала на синтетике, где известен
правильный ответ, и только потом на реальных данных, где его нет.

Что сделаем:

1. посчитаем значения Шепли по инструментам **точно**, перебрав все коалиции, — это AgentSHAP
   без Монте-Карло;
2. заменим перебор сэмплированием и посмотрим, сколько прогонов нужно и какой у оценки разброс;
3. увидим, чего метод не видит: бесполезный инструмент, взаимодействие двух инструментов
   и в чём он расходится с наивным «выключим по одному»;
4. посчитаем контрфактическую важность шагов рассуждения — Thought Anchors в миниатюре;
5. попробуем локализовать сбой в трейсе и поймём, почему у этой задачи такие низкие числа.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations, permutations
from math import factorial

rng = np.random.default_rng(0)

TOOLS = ["search", "db", "calc", "translate", "calendar"]


def agent_quality(subset):
    """Качество ответа агента при доступном наборе инструментов. Устройство известно:
    search обязателен — без него агент не отвечает вовсе;
    db и calc полезны по отдельности и дают ещё 0.25 вместе — это взаимодействие;
    calendar даёт немного, translate не даёт ничего."""
    S = set(subset)
    if "search" not in S:
        return 0.0
    q = 0.40
    q += 0.20 if "db" in S else 0.0
    q += 0.10 if "calc" in S else 0.0
    q += 0.25 if {"db", "calc"} <= S else 0.0
    q += 0.05 if "calendar" in S else 0.0
    return q


print("все инструменты:", agent_quality(TOOLS))
print("без search:  ", agent_quality([t for t in TOOLS if t != "search"]))
print("только search:", agent_quality(["search"]))

## 1. Точное значение Шепли: перебор коалиций

Формула та же, что в блоке про SHAP, только «признак» — это доступность инструмента.
Вклад инструмента $t$ — среднее по всем коалициям $S$, куда его добавляют:

$$\varphi_t = \sum_{S \subseteq N \setminus \{t\}} \frac{|S|!\,(n - |S| - 1)!}{n!}\,
\bigl[v(S \cup \{t\}) - v(S)\bigr]$$

Здесь $v(S)$ — качество ответа агента, которому доступны только инструменты из $S$. Каждое
такое $v(S)$ у настоящего агента — **полный прогон**, и именно поэтому перебор так дорог.

In [ ]:
def shapley_exact(tools, value):
    """Точные значения Шепли: перебор всех подмножеств."""
    n = len(tools)
    phi = {}
    for t in tools:
        rest = [x for x in tools if x != t]
        total = 0.0
        for k in range(len(rest) + 1):
            for S in combinations(rest, k):
                weight = factorial(k) * factorial(n - k - 1) / factorial(n)
                total += weight * (value(set(S) | {t}) - value(S))
        phi[t] = total
    return phi


phi = shapley_exact(TOOLS, agent_quality)
for tool, value in sorted(phi.items(), key=lambda kv: -kv[1]):
    print(f"{tool:10} {value:6.4f}")

print("\nкоалиций перебрано:", 2 ** len(TOOLS))
print("сумма вкладов =  ", round(sum(phi.values()), 6), "|", agent_quality(TOOLS))

Обратите внимание на последнюю строку: **сумма вкладов равна качеству полного набора**.
Это аксиома эффективности, и она же — самая быстрая проверка реализации. Если сумма не сходится
к $v(N) - v(\emptyset)$, реализация сломана.

Пять инструментов — это 32 коалиции, и перебор ещё возможен. Десять инструментов — 1024,
двадцать — миллион с лишним прогонов агента. Отсюда Монте-Карло.

## 2. Монте-Карло: сколько прогонов нужно

Оценка по перестановкам: берём случайный порядок инструментов, добавляем их по одному и
записываем прирост качества. Среднее по многим перестановкам сходится к значению Шепли.

Ниже — ошибка оценки в зависимости от числа перестановок, усреднённая по 20 независимым
запускам. Смотрите на наклон: ошибка падает как $1/\sqrt{n}$, то есть **точность вчетверо
дороже вдвое**.

In [ ]:
def shapley_mc(tools, value, n_samples, rng):
    """Оценка Шепли по случайным перестановкам инструментов."""
    phi = {t: 0.0 for t in tools}
    for _ in range(n_samples):
        order = list(rng.permutation(tools))
        before = set()
        v_before = value(before)
        for t in order:
            before.add(t)
            v_after = value(before)
            phi[t] += v_after - v_before
            v_before = v_after
    return {t: v / n_samples for t, v in phi.items()}


sizes = [10, 30, 100, 300, 1000, 3000]
errors = []
for n in sizes:
    runs = [shapley_mc(TOOLS, agent_quality, n, np.random.default_rng(seed)) for seed in range(20)]
    err = np.mean([max(abs(r[t] - phi[t]) for t in TOOLS) for r in runs])
    errors.append(err)
    print(f"{n:>5} {'перестановок → ошибка'} {err:.4f}")

plt.figure(figsize=(6, 3.4))
plt.plot(sizes, errors, marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("перестановок")
plt.ylabel("максимальная ошибка")
plt.title("Сходимость оценки Монте-Карло")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Теперь то, что авторы AgentSHAP проверяют отдельно и что стоит проверять всегда:
**устойчивость оценки между запусками**. Ниже — восемь независимых оценок на ста перестановках.

In [ ]:
runs = [shapley_mc(TOOLS, agent_quality, 100, np.random.default_rng(seed)) for seed in range(8)]
for tool in TOOLS:
    values = [r[tool] for r in runs]
    print(f"{tool:10} {np.mean(values):6.4f} ± {np.std(values):.4f}   "
          f"[{min(values):.3f}, {max(values):.3f}]")

Посмотрите на диапазоны `db` и `calc`: они пересекаются. На ста перестановках
единственный запуск вполне может поставить `calc` выше `db` — при том что точные значения
говорят обратное. Это и есть практический вывод: пока не измерен разброс, **разница между двумя
соседними инструментами может быть шумом сэмплирования**, а не свойством агента.

## 3. Чего метод не видит

Сравним значения Шепли с наивным «выключим по одному» — leave-one-out, разностью качества
полного набора и набора без инструмента.

In [ ]:
loo = {t: agent_quality(TOOLS) - agent_quality([x for x in TOOLS if x != t]) for t in TOOLS}

print(f"{'tool':10} {'Шепли':>10} {'leave-one-out':>10}")
for t in TOOLS:
    print(f"{t:10} {phi[t]:10.4f} {loo[t]:10.4f}")

print("\nсумма leave-one-out:", round(sum(loo.values()), 4))
print("сумма Шепли:       ", round(sum(phi.values()), 4))

Три вещи видны сразу.

**`translate` получил ровно ноль.** Это аксиома dummy: инструмент, который ничего не меняет ни в
одной коалиции, не получает вклада. Хорошая новость — метод не приписывает важность тому, что
просто присутствует в наборе.

**Сумма leave-one-out не равна качеству полного набора.** У Шепли равна, у leave-one-out — нет,
и разница ровно на величину взаимодействия. Наивный подход теряет то, что `db` и `calc` дают
только вместе: выключая по одному, вы не видите совместного эффекта.

**`search` получил больше половины.** Он обязателен: без него качество нулевое в любой коалиции.
Шепли честно отражает это, но заодно показывает ограничение метода — обязательный элемент
забирает вклад, который в каком-то смысле принадлежит всей системе.

## 4. Шаги рассуждения: Thought Anchors в миниатюре

Теперь другой игрок — не инструмент, а **шаг рассуждения**. Метод из работы: заменить шаг на
другой по смыслу и посмотреть, как изменилось распределение финальных ответов. Здесь трейс
фиксирован, а «модель» — симулятор, у которого известно, какой шаг насколько влияет на успех.

In [ ]:
TRACE = ["План: сначала уточнить период, потом считать", "Пользователь спрашивает про выручку", "Вызов db_lookup: выручка по кварталам", "Получены четыре числа", "Проверю, не путаю ли квартал с полугодием", "Ответ: рост на 12 процентов"]

# Влияние шага на успех — то, что метод должен восстановить
EFFECT = [0.45, 0.05, 0.30, 0.02, 0.10, 0.03]


def run_from(step_index, replaced, rng, n=400):
    """Доля успешных завершений, если шаг step_index заменён на другой по смыслу."""
    p = 0.9
    for i, effect in enumerate(EFFECT):
        if i == step_index and replaced:
            p -= effect
    p = float(np.clip(p, 0.0, 1.0))
    return rng.random(n) < p


base = run_from(-1, False, np.random.default_rng(0)).mean()
importance = []
for i in range(len(TRACE)):
    changed = run_from(i, True, np.random.default_rng(100 + i)).mean()
    importance.append(base - changed)
    print(f"{i + 1}. {TRACE[i][:46]:48} {base - changed:6.3f}")

plt.figure(figsize=(6.4, 3.2))
plt.barh(range(len(TRACE), 0, -1), importance)
plt.yticks(range(len(TRACE), 0, -1), [f"{i + 1}" for i in range(len(TRACE))])
plt.xlabel("падение доли успеха при замене шага")
plt.title("Контрфактическая важность шагов")
plt.tight_layout()
plt.show()

Наверх вышли первый и пятый шаги — планирование и проверка собственной неуверенности,
а не тот шаг, где выполняется вычисление. Это и есть находка работы: **якорями оказываются шаги,
где выбирается путь**, а не те, где считается ответ.

Обратите внимание на честность постановки: мы заменяем шаг на **другой по смыслу**. Если
подставлять переформулировку того же самого, метод измерит устойчивость к перефразированию,
а не влияние шага.

## 5. Локализация сбоя: почему числа такие низкие

Последняя задача — не объяснение, а диагностика: в трейсе мультиагентной системы найти, **кто**
из агентов и **на каком шаге** сломал прогон. Проверим две наивные эвристики на синтетических
трейсах, где виноватый известен.

In [ ]:
def make_trace(rng, n_agents=4, n_steps=12):
    """Трейс на 12 шагов и 4 агента; сбой внесён на известном шаге известным агентом."""
    culprit = int(rng.integers(n_agents))
    step = int(rng.integers(2, n_steps - 2))
    owners = rng.integers(0, n_agents, size=n_steps)
    owners[step] = culprit
    return owners, culprit, step


def guess_last(owners):
    """Наивная эвристика: виноват тот, кто говорил последним."""
    return int(owners[-1]), len(owners) - 1


def guess_middle(owners):
    """Наивная эвристика: сбой посередине трейса."""
    return int(owners[len(owners) // 2]), len(owners) // 2


rng2 = np.random.default_rng(7)
hits = {"last": [0, 0], "middle": [0, 0]}
N = 2000
for _ in range(N):
    owners, culprit, step = make_trace(rng2)
    for name, guess in (("last", guess_last), ("middle", guess_middle)):
        who, when = guess(owners)
        hits[name][0] += who == culprit
        hits[name][1] += when == step

for name, (who, when) in hits.items():
    print(f"{name:8} {'агент:'} {who / N:6.1%}   {'шаг:'} {when / N:6.1%}")

print(f"\n{'случайное угадывание агента:'} {1 / 4:.1%}")

Обе эвристики по агенту работают на уровне случайного угадывания, а по шагу —
практически никогда не попадают. Это и есть причина чисел из урока: у лучшего метода
в разборе Who&When **53,5 %** по агенту и **14,2 %** по шагу, а у сильных reasoning-моделей
на той же задаче — ниже 10 %. Задача «кто и когда» на порядок труднее задачи «что произошло»,
и никакой трейсер её сегодня не закрывает.

## Задания

Ответы отправьте на Степике — в шаге домашнего задания этого модуля.

1. Сколько коалиций пришлось бы перебрать для точного значения Шепли, будь у агента
   **восемь** инструментов?
2. Чему равно $\varphi$ инструмента `search`? Округлите до сотых.
3. Чему равна сумма всех значений Шепли и почему именно этому числу?
4. Добавьте в `agent_quality` шестой инструмент, который повторяет `db` (даёт ровно то же самое,
   когда `db` уже есть, и то же самое вместо него, когда `db` нет). Пересчитайте значения.
   Что произошло с вкладом `db` и почему это правильное поведение метода?
5. Замените в разделе 4 «замену шага на другой по смыслу» на «замену на переформулировку того же
   шага» — то есть сделайте эффект замены нулевым. Что покажет метод и почему это не значит,
   что шаги неважны?